In [2]:
import numpy as np
import tensorflow as tf
from keras import layers, models

In [3]:
sentences = "Your Data"

In [4]:
MAX_LEN = 100
VOCAB_SIZE = 15000
EMBEDDING_DIM = 512
NUM_HEADS = 8

vectorizer = layers.TextVectorization(
    max_tokens = VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN
)
vectorizer.adapt(sentences)
vocab = vectorizer.get_vocabulary()

In [5]:
class TokenAndPositionEmbedding(layers.Layer):
  def __init__(self, max_len, vocab_size, embed_dim):
    super().__init__()
    self.token_embedding = layers.Embedding(
        input_dim = vocab_size,
        output_dim = embed_dim
    )
    self.pos_embedding = layers.Embedding(
        input_dim = max_len,
        output_dim = embed_dim
    )

  def call(self, x):
    max_len = tf.shape(x)[-1]
    positions = tf.range(start = 0, limit = max_len, delta = 1)
    token_embed = self.token_embedding(x)         # TOKEN EMBEDDING
    pos_embed = self.pos_embedding(positions)     # POSITION EMBEDDING
    return token_embed + pos_embed

# ONE ENCODER BLOCK OF TRANFORMER

In [10]:
# INPUTS
inputs = layers.Input(shape = (MAX_LEN, ), dtype=tf.int32, name = 'INPUT')

# EMBEDDINGS
x_embed = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBEDDING_DIM)(inputs)

# MULTI-HEAD ATTENTION LAYER
mha_layer = layers.MultiHeadAttention(
    num_heads = NUM_HEADS,
    key_dim = EMBEDDING_DIM // NUM_HEADS,
    name = 'Multi-Head Attention'
)

attn_output, attn_score = mha_layer(query = x_embed, value = x_embed, return_attention_scores = True)

# RESIDUAL CONNETION 1
residual_output = x_embed + attn_output

# LAYER NORMALIZATION
normalizer = layers.LayerNormalization(
    epsilon=1e-6,
    name = "Normalizer"
)
x_contextual = normalizer(residual_output)

# FEED FORWARD NETWORK
x_ffn = layers.Dense(2048, activation = 'relu', name = 'FFN Layer 1')(x_contextual)
x_ffn = layers.Dense(EMBEDDING_DIM, activation = 'relu', name = 'FFN Layer 2')(x_ffn)

# RESIDUAL CONNETION 2
residual_output_2 = x_ffn + x_contextual

# LAYER NORMALIZATION - OUTPUT OF ENCODER
normalizer_2 = layers.LayerNormalization(
    epsilon = 1e-6,
    name = "Normalizer-2"
)

# OUTPUT
outputs = normalizer_2(residual_output_2)

encoder = models.Model(
    inputs = inputs,
    outputs = outputs
)

encoder.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ INPUT (InputLayer)  │ (None, 100)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_and_position… │ (None, 100, 512)  │  7,731,200 │ INPUT[0][0]       │
│ (TokenAndPositionE… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Multi-Head          │ [(None, 100,      │  1,050,624 │ token_and_positi… │
│ Attention           │ 512), (None, 8,   │            │ token_and_positi… │
│ (MultiHeadAttentio… │ 100, 100)]        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 100, 512)  │          0 │ token_and_positi… │
│                     │                   │            │ Multi-Head        │
│                     │                   │            │ Attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Normalizer          │ (None, 100, 512)  │      1,024 │ add_6[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ FFN Layer 1 (Dense) │ (None, 100, 2048) │  1,050,624 │ Normalizer[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ FFN Layer 2 (Dense) │ (None, 100, 512)  │  1,049,088 │ FFN Layer 1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_7 (Add)         │ (None, 100, 512)  │          0 │ FFN Layer         │
│                     │                   │            │ 2[0][0],          │
│                     │                   │            │ Normalizer[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Normalizer-2        │ (None, 100, 512)  │      1,024 │ add_7[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 10,883,584 (41.52 MB)

 Trainable params: 10,883,584 (41.52 MB)

 Non-trainable params: 0 (0.00 B)